In [ ]:
# AIRBORNE MICROPLASTIC DISPERSION MODEL
# 2D ADVECTION-DIFFUSION-DEPOSITION-SETTLING MODEL

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from mpl_toolkits.mplot3d import Axes3D


# ==========================================================
# Physical Parameters
# ==========================================================

# Domain (m)
Lx = 200
Ly = 200

Nx = 100
Ny = 100

x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)

dx = x[1] - x[0]
dy = y[1] - y[0]

X, Y = np.meshgrid(x, y)


# Wind velocity (m/s)
vx = 2.3
vy = 0.5

# Horizontal diffusion coefficients (m²/s)
Dx = 1.0
Dy = 1.0

# Deposition/decay coefficient (1/s)
lam = 0.005

# Mixing height (m)
H = 50


# Particle properties
rho_particle = 1100
rho_air = 1.204
g = 9.81
particle_diameter = 75e-6
mu = 1.81e-5

Vs = (
    (rho_particle - rho_air)
    * g
    * particle_diameter**2
    / (18 * mu)
)

print(f"Settling velocity = {Vs:.6e} m/s")


# ==========================================================
# Gaussian Emission Source
# ==========================================================

Q = 100
xf = 100
yf = 100
sigma = 2.5

Source = Q * np.exp(
    -((X - xf)**2 + (Y - yf)**2) / (2 * sigma**2)
)


# ==========================================================
# Initial Condition and Time
# ==========================================================

u0 = np.zeros((Ny, Nx))

dt = 0.25
T = 120
steps = int(T / dt)

print(f"Time step = {dt} s")
print(f"Simulation time = {T} s")
print(f"Number of time steps = {steps}")


# ==========================================================
# Numerical Solver
# ==========================================================

def advance_one_step(u):
    un = u.copy()
    new_u = un.copy()

    for j in range(1, Ny - 1):
        for i in range(1, Nx - 1):

            # Upwind advection
            dudx = (un[j, i] - un[j, i - 1]) / dx
            dudy = (un[j, i] - un[j - 1, i]) / dy

            # Central diffusion
            d2udx2 = (
                un[j, i + 1]
                - 2 * un[j, i]
                + un[j, i - 1]
            ) / dx**2

            d2udy2 = (
                un[j + 1, i]
                - 2 * un[j, i]
                + un[j - 1, i]
            ) / dy**2

            # Deposition and gravitational settling
            reaction = -(lam + Vs / H) * un[j, i]

            # Explicit finite-difference update
            new_u[j, i] = un[j, i] + dt * (
                -vx * dudx
                -vy * dudy
                +Dx * d2udx2
                +Dy * d2udy2
                +reaction
                +Source[j, i]
            )

    # Boundary conditions
    new_u[:, 0] = 0
    new_u[:, -1] = 0
    new_u[0, :] = 0
    new_u[-1, :] = 0

    return new_u


# ==========================================================
# Stability Check
# ==========================================================

CFL_x = abs(vx) * dt / dx
CFL_y = abs(vy) * dt / dy

diffusion_number = (
    Dx * dt / dx**2
    + Dy * dt / dy**2
)

print("\nStability Check")
print("-------------------------")
print(f"CFL_x = {CFL_x:.4f}")
print(f"CFL_y = {CFL_y:.4f}")
print(f"CFL_x + CFL_y = {CFL_x + CFL_y:.4f}")
print(f"Diffusion number = {diffusion_number:.4f}")

if CFL_x + CFL_y <= 1:
    print("Advection CFL condition: satisfied")
else:
    print("Warning: advection CFL condition may be violated")

if diffusion_number <= 0.5:
    print("Diffusion stability condition: satisfied")
else:
    print("Warning: diffusion stability condition may be violated")


# ==========================================================
# Run Simulation
# ==========================================================

u = u0.copy()

snapshots = []
snapshot_times = []

n_animation_frames = 120
snapshot_interval = max(
    1,
    int(np.ceil(steps / (n_animation_frames - 1)))
)

for n in range(steps + 1):

    if n % snapshot_interval == 0 or n == steps:
        snapshots.append(u.copy())
        snapshot_times.append(n * dt)

    if n < steps:
        u = advance_one_step(u)

u_final = u.copy()

print("\nSimulation completed.")
print(f"Final simulation time = {T} s")
print(f"Maximum final concentration = {u_final.max():.6f}")


# ==========================================================
# Animation
# ==========================================================

fig, ax = plt.subplots(figsize=(8, 6))

img = ax.imshow(
    snapshots[0],
    origin="lower",
    extent=[0, Lx, 0, Ly],
    cmap="plasma",
    vmin=0,
    vmax=max(20, u_final.max()),
    animated=True
)

plt.colorbar(
    img,
    ax=ax,
    label="Microplastic concentration"
)

ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.plot(xf, yf, "wo", markersize=8)


def update(frame):
    img.set_array(snapshots[frame])

    ax.set_title(
        f"Airborne Microplastic Dispersion\n"
        f"Time = {snapshot_times[frame]:.2f} s"
    )

    return img,


ani = FuncAnimation(
    fig,
    update,
    frames=len(snapshots),
    interval=40,
    blit=True
)

plt.close(fig)

HTML(ani.to_jshtml())


# ==========================================================
# Representative Concentration Fields
# ==========================================================

u_first = advance_one_step(u0.copy())

snapshot_fields = [
    (0.25, u_first),
    (T, u_final)
]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5),
    constrained_layout=True
)

vmax_snapshots = max(
    20,
    max(field.max() for _, field in snapshot_fields)
)

for ax_snap, (time_snap, field_snap) in zip(
    axes,
    snapshot_fields
):

    contour = ax_snap.contourf(
        X,
        Y,
        field_snap,
        50,
        cmap="plasma",
        vmin=0,
        vmax=vmax_snapshots
    )

    ax_snap.scatter(
        xf,
        yf,
        c="white",
        s=80,
        edgecolors="black",
        label="Emission Source"
    )

    ax_snap.set_xlabel("x (m)")
    ax_snap.set_ylabel("y (m)")
    ax_snap.set_title(
        f"Airborne Microplastic Concentration\n"
        f"t = {time_snap:.2f} s"
    )

fig.colorbar(
    contour,
    ax=axes,
    label="Microplastic concentration",
    shrink=0.85
)

fig.suptitle(
    "Representative Airborne Microplastic Dispersion",
    fontsize=14
)

plt.show()


# ==========================================================
# Final Airborne Microplastic Distribution
# ==========================================================

plt.figure(figsize=(8, 6))

plt.contourf(
    X,
    Y,
    u_final,
    50,
    cmap="plasma"
)

plt.colorbar(label="Concentration")

plt.scatter(
    xf,
    yf,
    c="white",
    s=100,
    edgecolors="black",
    label="Emission Source"
)

plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title(
    f"Final Airborne Microplastic Distribution "
    f"(t = {T:.1f} s)"
)

plt.legend()
plt.show()


# ==========================================================
# Centerline Concentration
# ==========================================================

plt.figure(figsize=(9, 5))

plt.plot(
    x,
    u_final[Ny // 2, :],
    linewidth=3
)

plt.xlabel("Distance (m)")
plt.ylabel("Concentration")
plt.title(
    f"Centerline Concentration "
    f"(t = {T:.1f} s)"
)

plt.grid(True)
plt.show()


# ==========================================================
# Three-Dimensional Final Concentration
# ==========================================================

fig = plt.figure(figsize=(10, 7))

ax = fig.add_subplot(
    111,
    projection="3d"
)

surf = ax.plot_surface(
    X,
    Y,
    u_final,
    cmap="plasma"
)

fig.colorbar(
    surf,
    ax=ax,
    shrink=0.6,
    label="Concentration"
)

ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_zlabel("Concentration")
ax.set_title(
    f"3D Final Airborne Microplastic Concentration "
    f"(t = {T:.1f} s)"
)

plt.show()


# ==========================================================
# Final Time Verification
# ==========================================================

print("\nFinal Time Verification")
print("-------------------------")
print(f"Animation final time = {snapshot_times[-1]:.2f} s")
print(f"Final solution time = {T:.2f} s")
print(f"Maximum final concentration = {u_final.max():.6f}")

if np.isclose(snapshot_times[-1], T):
    print("Final animation frame and final concentration correspond to 120 s.")
else:
    print("Warning: final animation frame does not correspond to 120 s.")